# Electivo de Bioinformática — Clase 9

## RNA-Seq II: expresión diferencial con DESeq2 y enriquecimiento de vías

**Programa:** Doctorado — 2º año
**Duración:** 2,5 horas

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Explicar por qué la normalización entre muestras usa los ***size factors*** de DESeq2 (mediana de razones) y no CPM/TPM.
2. Explorar las muestras con una **PCA** y elegir un **diseño** (`~ cell + dex`) acorde a lo que muestra.
3. Correr un análisis de **expresión diferencial** con DESeq2 e interpretar su tabla de resultados: `log2FoldChange`, `pvalue` y `padj`.
4. Visualizar los resultados (gráfico MA, volcán, recuentos de un gen).
5. Hacer un **análisis de sobrerrepresentación** (ORA): primero a mano para un término, y luego para todos los términos GO y vías KEGG con `clusterProfiler`.
6. Interpretar críticamente una lista de genes y de vías enriquecidas.

---
## 1. Dónde quedamos (5 min)

En la clase 8 vimos:

- Qué es un **recuento** y cómo `featureCounts` arma la tabla genes × muestras.
- Que un recuento depende de la expresión, pero también del **largo del gen** y de la **profundidad** de secuenciación.
- **CPM** (corrige profundidad) y **TPM** (corrige largo y profundidad; suma 10⁶ en cada muestra).

No alcanzamos a ver la última parte: por qué para comparar **entre muestras** se usa otra normalización (los *size factors* de DESeq2), y cómo se ven las muestras en una PCA. Partimos por ahí (secciones 4 y 5), porque es la base de lo que hace DESeq2, y luego pasamos a lo nuevo: **expresión diferencial** y **enriquecimiento de vías**.

Seguimos con los mismos datos: el estudio *airway* (4 líneas celulares de músculo liso de vía aérea, cada una sin tratar y tratada con **dexametasona**; 8 muestras).

---
## 2. Verificar el ambiente e instalar lo nuevo (10 min)

El ambiente `rnaseq` de la clase 8 ya tiene R, DESeq2 y los datos de *airway*. Verifiquemos:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate rnaseq

for HERR in R Rscript featureCounts; do
    if command -v "$HERR" >/dev/null 2>&1; then echo "OK    $HERR"; else echo "FALTA $HERR"; fi
done

Rscript -e 'for (p in c("DESeq2", "airway")) cat(if (requireNamespace(p, quietly = TRUE)) "OK   " else "FALTA", p, "\n")'

Si algo dice **FALTA**, revisen la sección 2 de la clase 8 (o su Apéndice, si `airway` no se pudo instalar).

Hoy necesitamos dos paquetes nuevos para el enriquecimiento:

| Paquete | Para qué |
|---|---|
| `bioconductor-clusterprofiler` | Análisis de enriquecimiento (GO, KEGG) y sus gráficos |
| `bioconductor-org.hs.eg.db` | Anotación del genoma humano: equivalencias entre identificadores (Ensembl, Entrez, símbolo) y términos GO de cada gen |

La instalación tarda varios minutos. **Abran una segunda terminal SSH** al servidor, corran ahí la instalación y sigan con la sección 3 en la primera terminal: no la vamos a necesitar hasta la sección 8.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda install -y -n rnaseq -c conda-forge -c bioconda \
    bioconductor-clusterprofiler bioconductor-org.hs.eg.db

---
## 3. Abrir R y preparar los datos (5 min)

Igual que en la clase 8: en la terminal, `conda activate rnaseq`, luego `R`, y peguen el contenido de cada celda `%%R` **sin la línea `%%R`**. La sesión de R conserva todo lo que vayan creando mientras no la cierren.

> Como en la clase 8 no alcanzamos a guardar el objeto de DESeq2, lo reconstruimos aquí (es rápido).

In [ ]:
%%R
suppressPackageStartupMessages(library(DESeq2))
library(ggplot2)
library(airway)
data(airway)

dir.create("~/bioinfo/clase9/resultados", recursive = TRUE, showWarnings = FALSE)
dir.create("~/bioinfo/clase9/graficos", showWarnings = FALSE)
setwd("~/bioinfo/clase9")

airway$dex <- relevel(airway$dex, "untrt")   # "sin tratar" = grupo de referencia
cts <- assay(airway)                         # recuentos crudos: genes x muestras

colData(airway)[, c("cell", "dex")]
round(colSums(cts) / 1e6, 1)                 # millones de fragmentos por muestra

Recuerden: las muestras van de **~15 a ~31 millones** de fragmentos, así que los recuentos crudos no son comparables entre muestras sin normalizar.

---
## 4. Normalizar entre muestras: el problema de composición y los *size factors* (15 min)

CPM y TPM dividen por el **total** de cada muestra. Eso funciona mal cuando unos pocos genes cambian mucho.

### 4.1 Un ejemplo de juguete: un solo gen cambia

Cuatro genes. Entre S1 y S2, **solo D cambia** (sube 7 veces); A, B y C están exactamente igual.

In [ ]:
%%R
comp <- matrix(c(100, 100, 100, 100,    # S1
                 100, 100, 100, 700),   # S2: solo D sube
               ncol = 2, dimnames = list(c("A", "B", "C", "D"), c("S1", "S2")))

# CPM
round(t(t(comp) / colSums(comp)) * 1e6)

Con CPM, **A, B y C parecen haber bajado 2,5 veces** en S2, aunque no cambiaron: como D se "llevó" más lecturas, a los demás les tocó una fracción menor del total. Esto es el **sesgo de composición**. En *airway*, los 100 genes más expresados se llevan ~30% de todas las lecturas, así que el problema es real.

### 4.2 La solución de DESeq2: mediana de razones

Supuesto: *la mayoría de los genes no cambia*. Entonces:

1. Para cada gen, se calcula una **referencia**: la media geométrica de sus recuentos en todas las muestras.
2. En cada muestra, se divide cada gen por su referencia → una **razón** por gen.
3. El ***size factor*** de la muestra es la **mediana** de esas razones (la mediana ignora a los pocos genes que cambian mucho).
4. Recuentos normalizados = recuentos / *size factor*.

In [ ]:
%%R
log_ref <- rowMeans(log(comp))           # paso 1 (en log, la media geométrica es una media simple)
razones <- exp(log(comp) - log_ref)      # paso 2
razones
sf_toy  <- apply(razones, 2, median)     # paso 3
sf_toy
t(t(comp) / sf_toy)                      # paso 4: solo D cambia

### 4.3 En los datos reales: a mano y con DESeq2

Con datos reales, los genes con algún 0 se excluyen del cálculo (su media geométrica sería 0).

In [ ]:
%%R
# A mano
log_ref <- rowMeans(log(cts))
usar    <- is.finite(log_ref)
sf_mano <- apply(cts[usar, ], 2, function(x) exp(median(log(x) - log_ref[usar])))

# Con DESeq2
dds <- DESeqDataSet(airway, design = ~ cell + dex)
dds <- estimateSizeFactors(dds)

round(rbind(a_mano = sf_mano, DESeq2 = sizeFactors(dds)), 3)

Las dos filas son **idénticas**: eso es lo que hace `estimateSizeFactors`. Y lo más importante para hoy: **DESeq2 guarda los recuentos crudos** y usa estos *size factors* **dentro** de su modelo estadístico. Por eso la entrada de una expresión diferencial son siempre **recuentos crudos**, nunca TPM ni CPM.

| Pregunta | Unidad adecuada |
|---|---|
| ¿Qué gen se expresa más **dentro de una muestra**? | **TPM** |
| ¿Cómo cambia **un gen entre muestras**? (para mirar o graficar) | **Recuentos normalizados** de DESeq2 |
| ¿Qué genes cambian **significativamente**? (prueba estadística) | **Recuentos crudos** + *size factors* dentro del modelo |

---
## 5. Mirar las muestras: PCA y elección del diseño (10 min)

Antes de buscar genes diferenciales, miramos las muestras en conjunto con una **PCA**. Primero transformamos los recuentos a una escala tipo log con `vst()` (*variance stabilizing transformation*); si no, los genes muy expresados dominarían todo.

In [ ]:
%%R
vsd <- vst(dds, blind = TRUE)
pca <- plotPCA(vsd, intgroup = c("dex", "cell"), returnData = TRUE)
pv  <- round(attr(pca, "percentVar") * 100)

pca[, c("PC1", "PC2", "dex", "cell")]
pv

pdf("graficos/pca.pdf", width = 6, height = 4.5)
print(ggplot(pca, aes(PC1, PC2, color = dex, shape = cell)) +
        geom_point(size = 4) +
        labs(x = paste0("PC1 (", pv[1], "%)"), y = paste0("PC2 (", pv[2], "%)")) +
        theme_bw())
dev.off()

(Para ver los PDF de hoy, descárguenlos con `scp` desde una terminal local, igual que en clases anteriores: `scp <usuario>@<servidor>:~/bioinfo/clase9/graficos/*.pdf .`)

- **PC1 (~41%)** separa **tratadas** (PC1 positivo) de **sin tratar** (negativo): la dexametasona es el efecto más grande.
- **PC2 (~26%)** separa las **líneas celulares**; la muestra tratada y la sin tratar de cada línea quedan a la misma altura.

Por eso el diseño es **`~ cell + dex`**: le decimos a DESeq2 que hay una diferencia de base entre líneas celulares, para que compare cada muestra tratada **con su propia línea sin tratar** (diseño pareado), en vez de confundir la variación entre líneas con el efecto del tratamiento. Por convención, **el factor de interés va al final** de la fórmula.

---
## 6. ¿Qué hace un análisis de expresión diferencial? (15 min)

Para **cada gen por separado**, DESeq2 se pregunta: *¿la diferencia entre tratadas y sin tratar es mayor que la que esperaríamos solo por variabilidad entre réplicas?*

En términos simples, para cada gen:

1. **Modela los recuentos** con una distribución **binomial negativa** (adecuada para recuentos que varían más que lo esperado por azar), usando los *size factors* para la profundidad.
2. **Estima la dispersión** (cuánto varía el gen entre réplicas). Con solo 4 réplicas por grupo esa estimación es poco precisa, así que DESeq2 "toma prestada" información de todos los genes con expresión parecida.
3. **Estima el cambio**: el **log2 fold change** (`log2FoldChange`) entre tratadas y sin tratar. 1 = el doble, 2 = cuatro veces, −1 = la mitad.
4. **Prueba** si ese cambio es distinto de 0 (prueba de Wald) → un **`pvalue`**.

### El problema de las pruebas múltiples

Vamos a hacer **~16.000 pruebas**, una por gen. Si usáramos `pvalue < 0,05`, aun si **ningún** gen cambiara, esperaríamos ~800 falsos positivos (5% de 16.000). Por eso se **corrige** el valor p: DESeq2 entrega **`padj`** (método de Benjamini-Hochberg), que controla la ***False Discovery Rate* (FDR)**: *de los genes que declaro significativos con `padj < 0,05`, se espera que como máximo ~5% sean falsos positivos*.

> **Regla práctica:** para decidir qué gen es "diferencial" se usa **`padj`**, nunca `pvalue`. Y significativo no es lo mismo que importante: un cambio de 10% puede ser muy significativo si el gen está muy expresado. Por eso se suele pedir además un tamaño de efecto mínimo (por ejemplo, `|log2FoldChange| > 1`, es decir, al menos el doble o la mitad).

---
## 7. Expresión diferencial con DESeq2 (25 min)

### 7.1 Filtrar genes con muy pocos recuentos

Genes con casi ninguna lectura no pueden dar resultados significativos y solo agregan pruebas (y más corrección por pruebas múltiples). Nos quedamos con los genes que tienen **al menos 10 recuentos en al menos 4 muestras** (4 = el tamaño del grupo más chico):

In [ ]:
%%R
keep <- rowSums(counts(dds) >= 10) >= 4
table(keep)
dds  <- dds[keep, ]

### 7.2 Correr DESeq2

Una sola función hace todo: *size factors*, dispersiones, modelo y prueba. Tarda alrededor de un minuto.

In [ ]:
%%R
dds <- DESeq(dds)
res <- results(dds, contrast = c("dex", "trt", "untrt"), alpha = 0.05)
res

`contrast = c("dex", "trt", "untrt")` significa: *factor `dex`, comparar `trt` contra `untrt`*. Un `log2FoldChange` **positivo** = el gen **sube** con dexametasona.

| Columna | Qué es |
|---|---|
| `baseMean` | Promedio de los recuentos normalizados en las 8 muestras |
| `log2FoldChange` | Cambio estimado (tratadas vs. sin tratar), en log2 |
| `lfcSE` | Error estándar de ese cambio (su incertidumbre) |
| `stat` | Estadístico de Wald (`log2FoldChange / lfcSE`) |
| `pvalue` | Valor p de la prueba |
| `padj` | Valor p ajustado por pruebas múltiples (FDR). `NA` = gen excluido automáticamente por baja expresión (*independent filtering*) |

In [ ]:
%%R
summary(res)

cat("padj < 0.05:                       ", sum(res$padj < 0.05, na.rm = TRUE), "\n")
cat("padj < 0.05 y log2FC >  1 (suben): ", sum(res$padj < 0.05 & res$log2FoldChange >  1, na.rm = TRUE), "\n")
cat("padj < 0.05 y log2FC < -1 (bajan): ", sum(res$padj < 0.05 & res$log2FoldChange < -1, na.rm = TRUE), "\n")

Deberían obtener ~**4.087** genes con `padj < 0,05` (2.228 suben, 1.859 bajan), pero solo ~**490** que suben y ~**456** que bajan al menos al doble. Esas dos listas de ~500 genes son las que usaremos para el enriquecimiento.

### 7.3 Los genes más significativos

Agregamos el **símbolo** del gen (más fácil de leer que el identificador de Ensembl) y ordenamos por `padj`:

In [ ]:
%%R
res$symbol <- rowData(dds)$gene_name
res_ord <- res[order(res$padj), ]
head(as.data.frame(res_ord)[, c("symbol", "baseMean", "log2FoldChange", "padj")], 15)

Arriba deberían aparecer genes como `SPARCL1`, `DUSP1`, `PER1`, `MAOA`, `GPX3`, la mayoría **subiendo** con dexametasona; `VCAM1` es uno de los que **bajan** (una molécula de adhesión inflamatoria, coherente con el efecto antiinflamatorio de los glucocorticoides).

Miremos los genes que seguimos en la clase 8:

In [ ]:
%%R
genes <- c(FKBP5 = "ENSG00000096060", TSC22D3 = "ENSG00000157514", ZBTB16 = "ENSG00000109906",
           CRISPLD2 = "ENSG00000103196", ANGPTL7 = "ENSG00000171819",
           ACTB = "ENSG00000075624", GAPDH = "ENSG00000111640")
round(as.data.frame(res[genes, c("baseMean", "log2FoldChange", "lfcSE", "padj")]), 4)

- `FKBP5` sube **2⁴ ≈ 16 veces** (`log2FoldChange` ≈ 4,0), `ZBTB16` más de 100 veces (≈ 7,3), `CRISPLD2` (el hallazgo principal del artículo original) ≈ 6 veces. Todos con `padj` extremadamente bajo.
- `GAPDH` **no** cambia (`padj` ≈ 0,40). Pero `ACTB`, otro gen "de mantención", **sí** cambia (log2FC ≈ 0,9, `padj` ≈ 0,008): no siempre se puede confiar en un solo gen como referencia.

### 7.4 Visualizar los resultados

In [ ]:
%%R
pdf("graficos/expresion_diferencial.pdf", width = 7, height = 5)

# 1) Gráfico MA: cambio vs. expresión promedio (azul = padj < 0.05)
plotMA(res, ylim = c(-6, 6), main = "Grafico MA")

# 2) Volcán: tamaño del cambio vs. significancia
sig <- !is.na(res$padj) & res$padj < 0.05 & abs(res$log2FoldChange) > 1
plot(res$log2FoldChange, -log10(res$padj), pch = 20, cex = 0.4,
     col = ifelse(sig, "firebrick", "grey60"),
     xlab = "log2 fold change (tratadas / sin tratar)", ylab = "-log10(padj)", main = "Volcan")
abline(v = c(-1, 1), h = -log10(0.05), lty = 2)

# 3) Recuentos normalizados de un gen, por muestra
plotCounts(dds, gene = genes["FKBP5"], intgroup = "dex", main = "FKBP5")

dev.off()

- **Gráfico MA:** cada punto es un gen; eje x = expresión promedio, eje y = log2FC. Los genes poco expresados tienen cambios más "ruidosos" (más dispersos) pero rara vez son significativos.
- **Volcán:** a la derecha arriba los que suben mucho y con mucha certeza; a la izquierda arriba los que bajan. Los genes en rojo cumplen `padj < 0,05` y `|log2FC| > 1`.
- **plotCounts:** siempre vale la pena mirar los recuentos de un gen antes de creerle a una tabla.

### 7.5 ¿Importó el diseño? `~ dex` vs. `~ cell + dex`

Repitamos el análisis ignorando la línea celular:

In [ ]:
%%R
dds_solo_dex <- DESeqDataSet(airway[keep, ], design = ~ dex)
dds_solo_dex <- DESeq(dds_solo_dex, quiet = TRUE)
res_solo_dex <- results(dds_solo_dex, contrast = c("dex", "trt", "untrt"), alpha = 0.05)

cat("~ dex        -> padj < 0.05:", sum(res_solo_dex$padj < 0.05, na.rm = TRUE), "\n")
cat("~ cell + dex -> padj < 0.05:", sum(res$padj < 0.05, na.rm = TRUE), "\n")

Con `~ dex` salen ~**2.773** genes; con `~ cell + dex`, ~**4.087**. Al incluir la línea celular, DESeq2 "descuenta" la variación entre líneas (lo que vimos en PC2), la variabilidad residual baja y el mismo efecto del tratamiento se detecta con más certeza. **El diseño se decide mirando el experimento (y la PCA), no después de ver cuál da más genes.**

### 7.6 Guardar los resultados

In [ ]:
%%R
write.csv(as.data.frame(res_ord), "resultados/deseq2_dex_trt_vs_untrt.csv")
saveRDS(dds, "resultados/dds_airway.rds")

---
## 8. Enriquecimiento de vías (30 min)

Tenemos ~500 genes que suben y ~450 que bajan. Mirarlos uno por uno no escala. La pregunta de un **análisis de enriquecimiento** es: *¿hay procesos biológicos o vías que aparezcan en mi lista **más de lo esperable por azar**?*

Necesitamos:
- **Conjuntos de genes** con significado biológico. Los más usados:
  - **Gene Ontology (GO)**: términos jerárquicos en tres categorías — proceso biológico (BP), función molecular (MF) y componente celular (CC). Hoy usamos **BP**.
  - **KEGG** / **Reactome**: vías metabólicas y de señalización curadas.
- Una **lista de genes de interés** (nuestros genes que suben).
- Un **universo**: todos los genes que **tuvieron la oportunidad** de salir en la lista = los genes que probamos en DESeq2 (no el genoma completo: un gen que no se expresa en músculo liso nunca podría aparecer, y contarlo inflaría artificialmente el enriquecimiento).

### 8.1 El test, a mano, para un solo término

Tomemos el término GO **"response to glucocorticoid"** (`GO:0051384`). Armamos una tabla de 2×2: ¿cuántos de nuestros genes que suben están en ese término, comparado con lo esperado según el universo?

In [ ]:
%%R
suppressPackageStartupMessages(library(org.Hs.eg.db))

universo <- rownames(res)[!is.na(res$padj)]                       # genes probados
suben    <- rownames(res)[which(res$padj < 0.05 & res$log2FoldChange >  1)]
bajan    <- rownames(res)[which(res$padj < 0.05 & res$log2FoldChange < -1)]

# Genes (Ensembl) anotados en el término, dentro del universo
termino  <- AnnotationDbi::select(org.Hs.eg.db, keys = "GO:0051384", keytype = "GOALL", columns = "ENSEMBL")
en_term  <- intersect(unique(termino$ENSEMBL), universo)

a <- sum(suben %in% en_term)            # suben y están en el término
b <- length(suben) - a                  # suben y no están
c <- length(en_term) - a                # no suben y están
d <- length(universo) - a - b - c       # no suben y no están
tabla <- matrix(c(a, c, b, d), nrow = 2,
                dimnames = list(c("en el término", "fuera del término"), c("suben", "no suben")))
tabla

cat("% del término entre los que suben:", round(100 * a / length(suben), 2), "\n")
cat("% del término en el universo:     ", round(100 * length(en_term) / length(universo), 2), "\n")
fisher.test(tabla, alternative = "greater")$p.value

Entre los genes que suben, ~**2%** pertenecen a "response to glucocorticoid"; en el universo, solo ~**0,5%**: una **sobrerrepresentación de ~4 veces**, con un valor p muy bajo (prueba exacta de Fisher, o su equivalente, la hipergeométrica). Eso es exactamente un **análisis de sobrerrepresentación (ORA)**. ¿Qué genes son?

In [ ]:
%%R
ids <- intersect(suben, en_term)
data.frame(symbol = res[ids, "symbol"], log2FC = round(res[ids, "log2FoldChange"], 2))

(Deberían aparecer, entre otros, `FAM107A`, `KLF9`, `IGFBP2` y `STC1`.)

### 8.2 Todos los términos GO a la vez con `clusterProfiler`

`enrichGO` hace la misma prueba para **miles** de términos GO y corrige por pruebas múltiples (`p.adjust`, igual que `padj`). Asegúrense de que la instalación de la sección 2 haya terminado.

In [ ]:
%%R
suppressPackageStartupMessages(library(clusterProfiler))

ego_suben <- enrichGO(gene = suben, universe = universo, OrgDb = org.Hs.eg.db,
                      keyType = "ENSEMBL", ont = "BP", pAdjustMethod = "BH", readable = TRUE)

head(as.data.frame(ego_suben)[, c("Description", "GeneRatio", "BgRatio", "p.adjust")], 15)

- **`GeneRatio`** = genes de la lista en el término / genes de la lista con anotación GO (en el ejemplo a mano, *a / (a + b)*).
- **`BgRatio`** = genes del universo en el término / genes del universo con anotación GO.
- El enriquecimiento es la comparación entre ambos; `p.adjust` dice si es significativo.

Deberían ver arriba términos como ***extracellular matrix organization***, ***cell-substrate adhesion*** y ***muscle system process***. Tiene sentido: son células de **músculo liso**, y los glucocorticoides modulan la matriz extracelular y la adhesión celular en la vía aérea.

Noten que varios términos son **casi lo mismo** (*extracellular matrix organization*, *extracellular structure organization*…): GO es jerárquico y un mismo grupo de genes cae en términos padres e hijos. `simplify(ego_suben)` agrupa los términos redundantes, si quieren una lista más corta.

Ahora los genes que **bajan**, y un gráfico de puntos para cada lista:

In [ ]:
%%R
ego_bajan <- enrichGO(gene = bajan, universe = universo, OrgDb = org.Hs.eg.db,
                      keyType = "ENSEMBL", ont = "BP", pAdjustMethod = "BH", readable = TRUE)
head(as.data.frame(ego_bajan)[, c("Description", "GeneRatio", "BgRatio", "p.adjust")], 10)

pdf("graficos/enriquecimiento_GO.pdf", width = 8, height = 6)
print(dotplot(ego_suben, showCategory = 15, title = "GO BP - genes que suben"))
print(dotplot(ego_bajan, showCategory = 15, title = "GO BP - genes que bajan"))
dev.off()

write.csv(as.data.frame(ego_suben), "resultados/GO_BP_suben.csv", row.names = FALSE)
write.csv(as.data.frame(ego_bajan), "resultados/GO_BP_bajan.csv", row.names = FALSE)

En el `dotplot`, cada fila es un término: el eje x es el `GeneRatio`, el tamaño del punto el número de genes y el color el `p.adjust`.

### 8.3 Vías KEGG

KEGG usa identificadores **Entrez** (números), así que primero convertimos de Ensembl a Entrez con `bitr`. Algunos genes no tienen equivalencia y se pierden en la conversión (es normal; `bitr` avisa cuántos). **`enrichKEGG` descarga las vías desde el sitio web de KEGG en el momento**, así que necesita conexión a internet desde el servidor.

In [ ]:
%%R
a_entrez <- function(ids) unique(bitr(ids, fromType = "ENSEMBL", toType = "ENTREZID", OrgDb = org.Hs.eg.db)$ENTREZID)

kegg_suben <- enrichKEGG(gene = a_entrez(suben), universe = a_entrez(universo), organism = "hsa")
kegg_suben <- setReadable(kegg_suben, OrgDb = org.Hs.eg.db, keyType = "ENTREZID")

head(as.data.frame(kegg_suben)[, c("Description", "GeneRatio", "BgRatio", "p.adjust", "geneID")], 10)

Si esta celda falla con un error de conexión, el servidor no tiene acceso a KEGG: sigan con el ejercicio usando solo los resultados de GO (la interpretación es la misma).

### 8.4 Una nota sobre GSEA

El ORA depende de un **corte arbitrario** (`padj < 0,05` y `|log2FC| > 1`): un gen con `log2FC = 0,99` queda fuera. La alternativa es **GSEA** (*Gene Set Enrichment Analysis*): ordena **todos** los genes probados según su cambio y pregunta si los genes de una vía tienden a acumularse arriba o abajo del ranking, sin cortes. En `clusterProfiler` existe como `gseGO()` y `gseKEGG()`. No lo veremos hoy, pero es bueno saber que existe y cuándo conviene.

---
## 9. Ejercicio integrador (en parejas, 20 min)

Respondan **en palabras**. Pueden usar las tablas y gráficos de hoy (si hace falta, vuelvan a imprimirlas en R).

1. **¿Qué significa el log2FC?** Calculen a mano el cambio de `FKBP5` en cada línea celular con los recuentos normalizados:
   ```r
   nc <- counts(dds, normalized = TRUE)
   tratadas   <- c("SRR1039509", "SRR1039513", "SRR1039517", "SRR1039521")
   sin_tratar <- c("SRR1039508", "SRR1039512", "SRR1039516", "SRR1039520")
   round(log2(nc[genes["FKBP5"], tratadas] / nc[genes["FKBP5"], sin_tratar]), 2)
   ```
   ¿Es consistente en las cuatro líneas? ¿Cómo se compara con el `log2FoldChange` que entregó DESeq2?

2. **Genes "de mantención".** `ACTB` sale significativo (`padj` ≈ 0,008) con log2FC ≈ 0,9. ¿Qué pasaría si se hubiera normalizado todo el experimento usando solo `ACTB` como referencia (como se hace a veces en qPCR)? ¿Por qué DESeq2 usa la mediana de miles de genes?

3. **Pensamiento crítico.** Una compañera propone hacer la expresión diferencial comparando con una prueba t los **TPM** de tratadas vs. sin tratar, y considerar diferenciales los genes con `pvalue < 0,05`. Den **tres** problemas de esta propuesta, basados en lo que vimos en las clases 8 y 9.

4. **Diseño.** Con `~ dex` salieron ~2.773 genes y con `~ cell + dex` ~4.087. Expliquen por qué, usando la PCA. Si en vez de 4 líneas celulares hubiéramos tenido 8 personas distintas (4 tratadas, 4 sin tratar, todas diferentes), ¿se podría usar el diseño `~ cell + dex`? ¿Qué se perdería?

5. **Interpretación del enriquecimiento.** El término "response to glucocorticoid" está enriquecido, pero **no** aparece entre los primeros de la lista de `enrichGO`. ¿Significa que el análisis falló? ¿Por qué creen que salen primero términos de matriz extracelular y adhesión? ¿Qué cambiaría si usáramos como universo **todos los genes del genoma** en vez de los genes probados?

---
## 10. Cierre y resumen (5 min)

Salgan de R con `q()` (respondan `n`: lo importante ya quedó guardado en `resultados/`).

### Comandos vistos hoy

```r
# Normalización entre muestras
dds <- DESeqDataSet(airway, design = ~ cell + dex)
dds <- estimateSizeFactors(dds);  sizeFactors(dds)

# Exploración
vsd <- vst(dds, blind = TRUE);    plotPCA(vsd, intgroup = c("dex", "cell"))

# Expresión diferencial
dds <- dds[rowSums(counts(dds) >= 10) >= 4, ]
dds <- DESeq(dds)
res <- results(dds, contrast = c("dex", "trt", "untrt"), alpha = 0.05)
summary(res);  plotMA(res);  plotCounts(dds, gene = "ENSG...", intgroup = "dex")

# Enriquecimiento (ORA)
fisher.test(tabla_2x2, alternative = "greater")
ego <- enrichGO(gene = lista, universe = universo, OrgDb = org.Hs.eg.db, keyType = "ENSEMBL", ont = "BP", readable = TRUE)
kk  <- enrichKEGG(gene = lista_entrez, universe = universo_entrez, organism = "hsa")
dotplot(ego, showCategory = 15)
```

### Ideas clave

- La entrada de DESeq2 son **recuentos crudos**; los *size factors* (mediana de razones) corrigen profundidad y composición dentro del modelo.
- El **diseño** se elige según el experimento: `~ cell + dex` compara cada línea consigo misma.
- Un gen es diferencial según **`padj`** (FDR), no `pvalue`; conviene exigir además un tamaño de efecto (`|log2FC| > 1`).
- El **ORA** pregunta si un conjunto de genes está sobrerrepresentado en la lista **respecto de un universo** (los genes probados), con una prueba hipergeométrica/Fisher por término y corrección por pruebas múltiples.
- Un resultado de enriquecimiento es una **hipótesis** para interpretar con la biología del sistema, no una conclusión.

### Lo que queda listo

- `~/bioinfo/clase9/resultados/deseq2_dex_trt_vs_untrt.csv`: la tabla completa de expresión diferencial, ordenada por `padj`.
- `~/bioinfo/clase9/resultados/GO_BP_suben.csv` y `GO_BP_bajan.csv`: los resultados del enriquecimiento.
- `~/bioinfo/clase9/resultados/dds_airway.rds`: el objeto de DESeq2 (se carga con `readRDS()`).

Las tablas `.csv` son un buen material para la próxima clase de **manipulación de tablas**.

### Tarea

1. Terminen las preguntas del ejercicio integrador.
2. Con `deseq2_dex_trt_vs_untrt.csv` y comandos de bash (`awk -F,`, `sort`, `head`): ¿cuáles son los 10 genes que **más bajan** (log2FC más negativo) entre los que tienen `padj < 0,05`? (Pista: las columnas son: identificador, `baseMean`, `log2FoldChange`, `lfcSE`, `stat`, `pvalue`, `padj`, `symbol`; recuerden saltarse el encabezado y que `padj` puede ser `NA`.) Elijan uno y busquen en la literatura si se ha descrito como regulado por glucocorticoides.